# Study WFS Mimic (v1)

**Author:** Aaron Roodman  
**Date Created:** 2026-06-04  
**Last Modified:** 2026-06-04  
**Status:** Draft  
**Keywords:** WFS, wavefront sensor, double Zernike, DOF recovery, FAM mimic  

## Description

Study whether the 4 corner wavefront sensors (WFS) alone can reconstruct
the optical state (degrees of freedom) that the full-array mode (FAM)
analysis provides.  Since actual WFS data is not available for the FAM
observations, we **mimic** WFS measurements by averaging FAM donut
Zernikes in annular wedge regions at the WFS field radius (~1.65 deg).

Key functionality:
1. Build per-visit "WFS mimic" Zernike vectors from FAM donut data
2. Subtract the measured intrinsic wavefront at the WFS positions
3. Build the OFC sensitivity matrix in the WFS-Zernike basis via SVD
4. Recover per-visit DOFs from the WFS mimic and compare against FAM DOFs

**Output:** Diagnostic plots, DOF comparison vs FAM analysis

**Based on:** `build_measured_intrinsic.ipynb`, `smatrix_doublez.ipynb`

## Change Log

| Date | Author | Description |
|------|--------|-------------|
| 2026-06-04 | Aaron Roodman | Initial version |

## Table of Contents

1. [Parameters](#params)
2. [Setup & Imports](#setup)
3. [Helper Functions](#functions)
4. [Data Loading](#data)
5. [Z4 Height Correction](#z4height)
6. [WFS Mimic Construction](#wfs-mimic)
7. [Sensitivity Matrix & SVD](#svd)
8. [DOF Recovery from WFS Mimic](#dof-recovery)
9. [Comparison: WFS vs FAM DOFs](#comparison)

<a id='params'></a>
## 1. Parameters

In [ ]:
# ============================================================
# Parameters — All configurable values collected here
# ============================================================

# ----- Input from build_measured_intrinsic -----
input_dir = 'output/build_measured_intrinsic/bin_2x_nkeep_34_OCS_riz_alt_55.0_75.0_rot_-3.0_3.0'
path_tag = 'pathA'

# ----- Binning selector (for donut parquet resolution) -----
binning = 'bin_2x'

# ----- Coordinate system -----
coord_sys = 'OCS'

# ----- WFS mimic geometry -----
# Annular wedge: radial range and azimuthal half-width
wfs_inner_radius_deg = 1.60     # inner edge of annular wedge
wfs_outer_radius_deg = 1.725    # outer edge of annular wedge
wfs_azimuth_width_deg = 7.5     # full azimuthal extent of each wedge (+/- 3.75 deg)

# Azimuthal offset for the 4 WFS positions.
# The 4 WFS centers are placed at delta + [0, 90, 180, 270] degrees
# on the focal plane.  Settable from -40 to +40 degrees.
delta_deg = 0.0

# ----- OFC sensitivity-matrix / SVD parameters -----
# Same defaults as build_measured_intrinsic.
k_min, k_max = 1, 6
n_keep = 34

# OFC config — auto-detected from RSP location.
ofc_config_subpath = 'ts_config_mttcs/MTAOS/v13/ofc'

# Path to the OFC normalization-weights yaml (50-DoF baseline).
# None -> auto-discover from $TS_CONFIG_MTTCS_DIR.
ofc_normalization_yaml = None

# Focal-plane Zernike normalization radius
fp_radius_basis = 1.75

# ----- Z4 CCD-height correction -----
height_map_fits = 'output/LSST_FP_cold_b_measurement_4col_bysurface.fits'
height_to_z4_factor = None    # None -> ccd_height.HEIGHT_TO_Z4_UM_PER_MM
intrinsic_transpose_bug = True

# ----- Output -----
output_dir = None

<a id='setup'></a>
## 2. Setup & Imports

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from astropy.table import QTable, vstack as table_vstack
from scipy.interpolate import RegularGridInterpolator
import yaml

# Make this notebook runnable from the aos/ directory.
if not (Path.cwd() / 'code' / 'measured_intrinsic.py').exists():
    for _anc in Path.cwd().parents:
        if (_anc / 'code' / 'measured_intrinsic.py').exists():
            os.chdir(_anc)
            print(f'(chdir -> {_anc})')
            break

sys.path.insert(0, str(Path.cwd().parent))
from common.utils import setup_plotting, detect_rsp_location, get_packages_dir
from common.zernike_names import NOLL_NAMES

setup_plotting()

sys.path.insert(0, 'code')
from dz_fitting import derive_noll_indices, focal_plane_zernike_basis
from measured_intrinsic import (
    apply_visit_filters,
    bin_median_focal,
    interpolate_grid_at_donuts,
)
from intrinsics_lib import (
    build_visit_marker_lookup, visit_marker_style,
)
from run_pipeline import (
    load_runs as _rp_load_runs,
    load_param_sets as _rp_load_param_sets,
    resolve_run as _rp_resolve_run,
    donut_path as _rp_donut_path,
)

# OFC / ts_ofc imports (RSP only)
from lsst.ts.ofc import OFCData, SensitivityMatrix

# CCD height-map helpers (RSP only)
from ccd_height import (
    compute_fp_coords,
    make_metrology_table,
    get_height_interpolator,
    height_to_z4,
    transpose_around_ccd_centers,
    HEIGHT_TO_Z4_UM_PER_MM,
)
from lsst.obs.lsst import LsstCam

print('All imports OK.')

<a id='functions'></a>
## 3. Helper Functions

### MeasuredIntrinsicGrid

A class that reads the measured-intrinsic grid parquet produced by
`build_measured_intrinsic` and provides bilinear interpolation of the
intrinsic Zernikes at arbitrary (thx_deg, thy_deg) positions in the
OCS coordinate system.  This is used to evaluate the intrinsic wavefront
at the WFS mimic positions.

In [ ]:
class MeasuredIntrinsicGrid:
    """Interpolator for the measured-intrinsic wavefront grid.

    Reads the grid parquet (from build_measured_intrinsic) and builds
    one RegularGridInterpolator per Zernike index.  Provides
    `evaluate(thx_deg, thy_deg)` returning shape (n_points, n_zk).
    """

    def __init__(self, grid_parquet_path):
        tbl = QTable.read(str(grid_parquet_path), format='parquet')
        self.coord_sys = str(tbl['coord_sys'][0])
        self.nollIndices = np.array(tbl['nollIndices'][0])
        self.iZs = [int(n) for n in self.nollIndices]
        self.iZidx = {iZ: i for i, iZ in enumerate(self.iZs)}
        self.n_zk = len(self.iZs)

        thx_all = np.array(tbl['thx_deg'], dtype=float)
        thy_all = np.array(tbl['thy_deg'], dtype=float)
        zk_all = np.array([np.array(row) for row in tbl['zk']])

        thx_unique = np.sort(np.unique(np.round(thx_all, 8)))
        thy_unique = np.sort(np.unique(np.round(thy_all, 8)))
        self.thx_grid = thx_unique
        self.thy_grid = thy_unique
        nx, ny = len(thx_unique), len(thy_unique)

        self._interps = []
        for zi in range(self.n_zk):
            grid_2d = np.full((nx, ny), np.nan)
            for row_idx in range(len(tbl)):
                xi = np.searchsorted(thx_unique, np.round(thx_all[row_idx], 8))
                yi = np.searchsorted(thy_unique, np.round(thy_all[row_idx], 8))
                grid_2d[xi, yi] = zk_all[row_idx, zi]

            nan_mask = np.isnan(grid_2d)
            if nan_mask.any():
                grid_2d[nan_mask] = 0.0

            interp = RegularGridInterpolator(
                (thx_unique, thy_unique), grid_2d,
                method='linear', bounds_error=False, fill_value=np.nan)
            self._interps.append(interp)

        print(f'MeasuredIntrinsicGrid: {nx}x{ny} grid, '
              f'{self.n_zk} Zernikes, coord_sys={self.coord_sys}')

    def evaluate(self, thx_deg, thy_deg):
        """Interpolate intrinsic Zernikes at given positions.

        Parameters
        ----------
        thx_deg, thy_deg : array-like
            Field angles in degrees.

        Returns
        -------
        zk : ndarray, shape (n_points, n_zk)
        """
        thx = np.atleast_1d(np.asarray(thx_deg, dtype=float))
        thy = np.atleast_1d(np.asarray(thy_deg, dtype=float))
        pts = np.column_stack((thx, thy))
        result = np.column_stack([interp(pts) for interp in self._interps])
        return result

### WFS Mimic Measurement

For each visit, select donuts in 4 annular wedge regions at the WFS
field radius.  The wedges are centered at azimuthal angles
`delta + [0, 90, 180, 270]` degrees, with a configurable radial range
and azimuthal width.  The median Zernike over each wedge is the
"WFS mimic" measurement for that position.

In [ ]:
def mimic_wfs_measurement(donut_df, visit_mask, coord_sys,
                          wfs_inner_radius_deg, wfs_outer_radius_deg,
                          wfs_azimuth_width_deg, delta_deg):
    """Build 4 WFS-mimic Zernike vectors from donuts in annular wedges.

    Parameters
    ----------
    donut_df : DataFrame
        Full donut table (all visits).
    visit_mask : ndarray of bool
        Mask selecting donuts for this visit.
    coord_sys : str
        'OCS' or 'CCS'.
    wfs_inner_radius_deg, wfs_outer_radius_deg : float
        Radial bounds of the annular wedge (degrees).
    wfs_azimuth_width_deg : float
        Full azimuthal extent of each wedge (degrees).
    delta_deg : float
        Azimuthal offset of the wedge centers.

    Returns
    -------
    dict with keys:
        'thx_deg', 'thy_deg' : ndarray (4,) — wedge center positions
        'zk_median' : ndarray (4, n_zk) — median Zernike per wedge
        'n_donuts' : ndarray (4,) — donut count per wedge
    """
    thx_col = f'thx_{coord_sys}'
    thy_col = f'thy_{coord_sys}'
    zk_col = f'zk_{coord_sys}'

    thx_rad = np.asarray(donut_df.loc[visit_mask, thx_col], dtype=float)
    thy_rad = np.asarray(donut_df.loc[visit_mask, thy_col], dtype=float)
    thx_deg = np.degrees(thx_rad)
    thy_deg = np.degrees(thy_rad)
    zk_arr = np.stack(donut_df.loc[visit_mask, zk_col].values).astype(float)

    r_deg = np.hypot(thx_deg, thy_deg)
    az_deg = np.degrees(np.arctan2(thy_deg, thx_deg)) % 360.0

    wfs_centers_az = [(delta_deg + offset) % 360.0 for offset in [0, 90, 180, 270]]
    half_az = wfs_azimuth_width_deg / 2.0

    n_zk = zk_arr.shape[1]
    zk_median = np.full((4, n_zk), np.nan)
    thx_out = np.zeros(4)
    thy_out = np.zeros(4)
    n_donuts = np.zeros(4, dtype=int)

    radial_mask = (r_deg >= wfs_inner_radius_deg) & (r_deg <= wfs_outer_radius_deg)

    for i, az_center in enumerate(wfs_centers_az):
        az_lo = (az_center - half_az) % 360.0
        az_hi = (az_center + half_az) % 360.0
        if az_lo < az_hi:
            az_mask = (az_deg >= az_lo) & (az_deg <= az_hi)
        else:
            az_mask = (az_deg >= az_lo) | (az_deg <= az_hi)

        wedge_mask = radial_mask & az_mask
        n_donuts[i] = int(wedge_mask.sum())

        r_mid = 0.5 * (wfs_inner_radius_deg + wfs_outer_radius_deg)
        az_rad = np.radians(az_center)
        thx_out[i] = r_mid * np.cos(az_rad)
        thy_out[i] = r_mid * np.sin(az_rad)

        if n_donuts[i] >= 3:
            zk_median[i] = np.nanmedian(zk_arr[wedge_mask], axis=0)

    return {
        'thx_deg': thx_out,
        'thy_deg': thy_out,
        'zk_median': zk_median,
        'n_donuts': n_donuts,
    }

### Z4 Height Correction for WFS Mimic

The Z4 (defocus) Zernike has a contribution from the physical CCD height
variation across the focal plane.  For donuts that have different intra-
and extra-focal positions, we compute the Z4 height at both positions
and average them.  This is the correct treatment because the donut pair
measurement is sensitive to the average defocus at the two positions.

In [ ]:
def compute_z4_height_correction(donut_df, visit_mask, interp, camera,
                                  factor=HEIGHT_TO_Z4_UM_PER_MM,
                                  intrinsic_transpose_bug=True):
    """Compute per-donut Z4 height correction averaging intra+extra positions.

    Returns
    -------
    z4hgt : ndarray (n_donuts_in_mask,)
        Z4 height contribution (μm) — average of intra and extra.
    z4hgt_intrinsic : ndarray (n_donuts_in_mask,)
        Z4 height as used by the pipeline intrinsic (with or without
        the per-CCD transpose bug).
    """
    sub = donut_df.loc[visit_mask]

    # Focal-plane coords for intra and extra separately
    fpx_intra, fpy_intra = compute_fp_coords(
        sub, camera, x_col='centroid_x_intra', y_col='centroid_y_intra')
    fpx_extra, fpy_extra = compute_fp_coords(
        sub, camera, x_col='centroid_x_extra', y_col='centroid_y_extra')

    z4_intra = height_to_z4(interp(fpx_intra, fpy_intra), factor=factor)
    z4_extra = height_to_z4(interp(fpx_extra, fpy_extra), factor=factor)
    z4hgt = 0.5 * (z4_intra + z4_extra)

    if intrinsic_transpose_bug:
        det_names = np.asarray(sub['detector']).astype(str)
        fpx_t_intra, fpy_t_intra = transpose_around_ccd_centers(
            fpx_intra, fpy_intra, det_names, camera)
        fpx_t_extra, fpy_t_extra = transpose_around_ccd_centers(
            fpx_extra, fpy_extra, det_names, camera)
        z4_t_intra = height_to_z4(interp(fpx_t_intra, fpy_t_intra), factor=factor)
        z4_t_extra = height_to_z4(interp(fpx_t_extra, fpy_t_extra), factor=factor)
        z4hgt_intrinsic = 0.5 * (z4_t_intra + z4_t_extra)
    else:
        z4hgt_intrinsic = z4hgt

    return z4hgt, z4hgt_intrinsic

### DOF Recovery

Given U-mode amplitudes from the WFS SVD, recover physical degrees of
freedom using the V matrix and singular values, following the same
approach as `build_measured_intrinsic`.

In [ ]:
LABELS_50DOF = [
    'M2_dz', 'M2_dx', 'M2_dy', 'M2_rx', 'M2_ry',
    'Cam_dz', 'Cam_dx', 'Cam_dy', 'Cam_rx', 'Cam_ry',
    'B1_1', 'B1_2', 'B1_3', 'B1_4', 'B1_5',
    'B1_6', 'B1_7', 'B1_8', 'B1_9', 'B1_10',
    'B1_11', 'B1_12', 'B1_13', 'B1_14', 'B1_15',
    'B1_16', 'B1_17', 'B1_18', 'B1_19', 'B1_20',
    'B2_1', 'B2_2', 'B2_3', 'B2_4', 'B2_5',
    'B2_6', 'B2_7', 'B2_8', 'B2_9', 'B2_10',
    'B2_11', 'B2_12', 'B2_13', 'B2_14', 'B2_15',
    'B2_16', 'B2_17', 'B2_18', 'B2_19', 'B2_20',
]
DOF_UNITS_50 = (
    ['μm', 'μm', 'μm', 'arcsec', 'arcsec'] +
    ['μm', 'μm', 'μm', 'arcsec', 'arcsec'] +
    ['μm'] * 20 +
    ['μm'] * 20
)


def recover_dof_per_visit(A_modes, V, Sigma, N_diag, n_keep):
    """Recover physical DOF estimates per visit from U-mode amplitudes.

    Parameters
    ----------
    A_modes : ndarray (n_visits, n_keep)
        U-mode amplitudes per visit (= U_eff^T @ w).
    V : ndarray (n_dof, n_dof)
        Right singular vectors of S.
    Sigma : ndarray (n_dof,)
        Singular values.
    N_diag : ndarray (n_dof,)
        OFC normalization weights (diagonal of N).
    n_keep : int

    Returns
    -------
    dof : ndarray (n_visits, n_dof)
        Physical DOF (mixed units per DOF_UNITS_50).
    """
    V_eff = V[:, :n_keep]
    inv_sig = 1.0 / Sigma[:n_keep]
    A = np.where(np.isfinite(A_modes), A_modes, 0.0)
    dof_norm = (V_eff * inv_sig[None, :]) @ A.T
    dof_phys = N_diag[:, None] * dof_norm
    return dof_phys.T

### Donut Parquet Loading

Resolve and load the multi-chunk donut parquet files, using the same
infrastructure as `build_measured_intrinsic`.

In [ ]:
def resolve_chunk_parquets(binning, donut_parquets=None):
    """Return the list of donut parquet paths for the given binning."""
    if donut_parquets:
        return [Path(p) for p in donut_parquets]
    target_ps = f'fam_danish_v1_triplets_{binning}'
    runs = _rp_load_runs().get('runs', {})
    param_sets = _rp_load_param_sets()
    paths = []
    for name, cfg in runs.items():
        if cfg.get('param_set') != target_ps:
            continue
        resolved = _rp_resolve_run(cfg, param_sets)
        p = Path(_rp_donut_path(resolved))
        if p.exists():
            paths.append(p)
        else:
            print(f'  (skipping {name}: {p} not found)')
    if not paths:
        raise FileNotFoundError(
            f'No donut parquet files resolved for binning={binning!r}')
    return paths


def visits_sidecar_path(donut_parquet_path):
    p = Path(donut_parquet_path)
    return p.with_name(p.stem + '_visits.parquet')


def load_chunks(donut_parquet_paths):
    """Read donut + visits sidecars across chunks."""
    donut_frames = []
    visits_tables = []
    for p in donut_parquet_paths:
        vp = visits_sidecar_path(p)
        assert p.exists(), f'missing {p}'
        assert vp.exists(), f'missing {vp}'
        d = pd.read_parquet(p)
        v = QTable.read(str(vp), format='parquet')
        donut_frames.append(d)
        visits_tables.append(v)
        print(f'  {p.name}:  {len(d):>8d} donuts,  {len(v):>4d} visits')
    donut_df = pd.concat(donut_frames, ignore_index=True)
    visits_full = (visits_tables[0]
                   if len(visits_tables) == 1
                   else table_vstack(visits_tables, join_type='exact'))
    return donut_df, visits_full


def filter_donuts_by_visits(donut_df, visits_kept):
    """Return rows of donut_df whose (day_obs, seq_num) appears in visits_kept."""
    keep_keys = set(zip(
        np.asarray(visits_kept['day_obs']).tolist(),
        np.asarray(visits_kept['seq_num']).tolist(),
    ))
    keys = list(zip(donut_df['day_obs'].tolist(),
                    donut_df['seq_num'].tolist()))
    mask = np.array([k in keep_keys for k in keys])
    return donut_df.loc[mask].reset_index(drop=True)

<a id='data'></a>
## 4. Data Loading

Load three data sources:
1. **DZ fits parquet** from `build_measured_intrinsic` — per-visit FAM
   analysis results including DZ coefficients, u/v-mode amplitudes, and
   recovered DOFs.  These are the "truth" for comparison.
2. **Measured intrinsic grid parquet** — the focal-plane wavefront
   intrinsic on a regular grid, used to subtract the intrinsic from the
   WFS mimic measurements.
3. **Donut parquet files** — the raw per-donut Zernike measurements from
   FAM, which we average in WFS-like regions.

In [ ]:
# --- Load FAM analysis results ---
input_path = Path(input_dir)
dz_fits_path = sorted(input_path.glob(f'*_{path_tag}_dz_fits.parquet'))
grid_path = sorted(input_path.glob(f'*_{path_tag}_grid.parquet'))

assert len(dz_fits_path) == 1, f'Expected 1 dz_fits file, found {len(dz_fits_path)}'
assert len(grid_path) == 1, f'Expected 1 grid file, found {len(grid_path)}'

dz_df = pd.read_parquet(dz_fits_path[0])
print(f'FAM dz_fits: {len(dz_df)} visits, {len(dz_df.columns)} columns')
print(f'  day_obs range: {dz_df["day_obs"].min()} - {dz_df["day_obs"].max()}')

intrinsic_grid = MeasuredIntrinsicGrid(grid_path[0])

In [ ]:
# --- Load donut parquets ---
donut_parquet_paths = resolve_chunk_parquets(binning)
print(f'Using binning = {binning!r};  {len(donut_parquet_paths)} chunk(s):')
for p in donut_parquet_paths:
    print(f'   {p}')

donut_full, visits_full = load_chunks(donut_parquet_paths)
print(f'\nTotal: {len(donut_full)} donuts, {len(visits_full)} visits')

# Filter to visits that are in the FAM analysis
fam_keys = set(zip(dz_df['day_obs'].tolist(), dz_df['seq_num'].tolist()))
visits_kept_mask = np.array([
    (int(d), int(s)) in fam_keys
    for d, s in zip(np.asarray(visits_full['day_obs']),
                    np.asarray(visits_full['seq_num']))
])
visits_kept = visits_full[visits_kept_mask]
print(f'Visits matching FAM analysis: {len(visits_kept)}/{len(visits_full)}')

donut_df = filter_donuts_by_visits(donut_full, visits_kept)
print(f'Donuts after filter: {len(donut_df)}/{len(donut_full)}')

# Derive Noll indices
nZk = np.stack(donut_df[f'zk_{coord_sys}'].values).shape[1]
noll_arr = (np.array(visits_kept['nollIndices'][0])
            if 'nollIndices' in visits_kept.colnames else None)
iZs, iZidx = derive_noll_indices(nZk, noll_arr)
iZs_arr = np.array(iZs, dtype=int)
n_j = len(iZs_arr)
print(f'Pupil Noll indices ({n_j}): {iZs_arr}')

<a id='z4height'></a>
## 5. Z4 Height Correction

The Z4 (defocus) Zernike includes a contribution from the physical
height variation of CCDs across the focal plane.  We precompute the Z4
height correction for every donut, averaging the height at the intra-
and extra-focal donut positions.  This correction is applied to the
*measured* Zernike before comparing against the intrinsic (which already
has the height contribution baked in, including any pipeline bugs).

In [ ]:
camera = LsstCam.getCamera()

z4_can_correct = (height_map_fits
                  and Path(height_map_fits).exists()
                  and 4 in iZs)

Z4hgt = None
Z4hgt_intrinsic = None

if z4_can_correct:
    print(f'Loading metrology: {height_map_fits}')
    metrology = make_metrology_table(height_map_fits)
    print(f'  {len(metrology)} metrology points across '
          f'{len(set(metrology["det"]))} sensors')

    factor = (height_to_z4_factor
              if height_to_z4_factor is not None
              else HEIGHT_TO_Z4_UM_PER_MM)
    interp_height = get_height_interpolator(metrology)

    # Per-donut Z4 height correction, averaging intra + extra positions
    Z4hgt, Z4hgt_intrinsic = compute_z4_height_correction(
        donut_df, np.ones(len(donut_df), dtype=bool),
        interp_height, camera,
        factor=factor,
        intrinsic_transpose_bug=intrinsic_transpose_bug)

    print(f'Z4hgt (intra+extra avg): '
          f'mean={np.nanmean(Z4hgt):+.4f} μm, '
          f'std={np.nanstd(Z4hgt):.4f} μm')
    print(f'Z4hgt_intrinsic:         '
          f'mean={np.nanmean(Z4hgt_intrinsic):+.4f} μm, '
          f'std={np.nanstd(Z4hgt_intrinsic):.4f} μm')
else:
    print('Skipping Z4 height correction')

<a id='wfs-mimic'></a>
## 6. WFS Mimic Construction

For each visit, we:
1. Select donuts in 4 annular wedge regions at the WFS field radius
2. Compute the median Zernike across each wedge
3. Apply Z4 height correction if available
4. Subtract the measured intrinsic at the wedge center positions
5. The result is a "WFS deviation" vector per visit — the signal that
   the AOS control loop would act on

The WFS deviation captures both the correctable optical state (DOFs)
and any uncorrectable residual.  A key question is how well the DOFs
can be recovered from just 4 field positions.

In [ ]:
# Build per-visit WFS mimic measurements
dobs_arr = np.asarray(donut_df['day_obs'])
snum_arr = np.asarray(donut_df['seq_num'])

# Sort dz_df to ensure consistent ordering
dz_df = dz_df.sort_values(['day_obs', 'seq_num']).reset_index(drop=True)
n_visits = len(dz_df)

wfs_results = []
for v_idx in range(n_visits):
    d = int(dz_df.iloc[v_idx]['day_obs'])
    s = int(dz_df.iloc[v_idx]['seq_num'])
    visit_mask = (dobs_arr == d) & (snum_arr == s)

    result = mimic_wfs_measurement(
        donut_df, visit_mask, coord_sys,
        wfs_inner_radius_deg, wfs_outer_radius_deg,
        wfs_azimuth_width_deg, delta_deg)
    result['day_obs'] = d
    result['seq_num'] = s
    wfs_results.append(result)

    if v_idx == 0:
        print(f'Visit {d}/{s}: donuts per wedge = {result["n_donuts"]}')

# Summary
all_counts = np.array([r['n_donuts'] for r in wfs_results])
print(f'\n{n_visits} visits processed.')
print(f'Donuts per wedge — min: {all_counts.min(axis=0)}, '
      f'median: {np.median(all_counts, axis=0).astype(int)}, '
      f'max: {all_counts.max(axis=0)}')
low_count = np.any(all_counts < 3, axis=1)
print(f'Visits with <3 donuts in any wedge: {low_count.sum()}/{n_visits}')

### WFS Mimic Visualization

Show the donut distribution and WFS wedge regions for an example visit.

In [ ]:
from matplotlib.patches import Polygon


def annular_wedge_verts(az_center_deg, half_az_deg, r_in, r_out, n=60):
    """Vertices (in plot coords x=thy, y=thx) of an annular wedge outline.

    Azimuth is defined as in the mimic selection: az = atan2(thy, thx),
    so a point at azimuth ``a`` and radius ``r`` plots at
    (x=thy=r*sin(a), y=thx=r*cos(a)).
    """
    az = np.radians(np.linspace(az_center_deg - half_az_deg,
                                az_center_deg + half_az_deg, n))
    x_in, y_in = r_in * np.sin(az), r_in * np.cos(az)
    x_out, y_out = r_out * np.sin(az), r_out * np.cos(az)
    xs = np.concatenate([x_in, x_out[::-1]])
    ys = np.concatenate([y_in, y_out[::-1]])
    return np.column_stack([xs, ys])


# Pick an example visit where all 4 wedges are populated (>=3 donuts),
# so the validation plot shows a representative, fully-sampled case.
all_counts = np.array([r['n_donuts'] for r in wfs_results])
_good_visits = np.where(np.all(all_counts >= 3, axis=1))[0]
ex_idx = int(_good_visits[0]) if len(_good_visits) else 0

ex = wfs_results[ex_idx]
d, s = ex['day_obs'], ex['seq_num']
visit_mask = (dobs_arr == d) & (snum_arr == s)
thx_v = np.degrees(np.asarray(donut_df.loc[visit_mask, f'thx_{coord_sys}'], dtype=float))
thy_v = np.degrees(np.asarray(donut_df.loc[visit_mask, f'thy_{coord_sys}'], dtype=float))

fig, ax = plt.subplots(figsize=(8, 8), layout='constrained')
ax.plot(thy_v, thx_v, '.', ms=1, alpha=0.3, color='gray', label='all donuts')

colors = ['tab:red', 'tab:blue', 'tab:green', 'tab:orange']
half_az = wfs_azimuth_width_deg / 2.0
wfs_centers_az = [(delta_deg + offset) % 360.0 for offset in [0, 90, 180, 270]]

for i in range(4):
    # Open wedge showing the exact (radial x azimuthal) selection region
    verts = annular_wedge_verts(wfs_centers_az[i], half_az,
                                wfs_inner_radius_deg, wfs_outer_radius_deg)
    ax.add_patch(Polygon(verts, closed=True, fill=False,
                         edgecolor=colors[i], lw=1.8))
    # Wedge center marker
    ax.plot(ex['thy_deg'][i], ex['thx_deg'][i], 'o', ms=10,
            color=colors[i], label=f'WFS {i} (n={ex["n_donuts"][i]})')

ax.set_xlabel(f'thy_{coord_sys} [deg]')
ax.set_ylabel(f'thx_{coord_sys} [deg]')
ax.set_title(f'WFS Mimic Regions — visit {d}/{s}, delta={delta_deg:.0f}°')
ax.set_aspect('equal')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.show()

### Compute WFS Deviation Vectors

Subtract the measured intrinsic wavefront (from the grid) at the WFS
positions to get the deviation — this is the "measurement" that the
OFC would use to correct the optical state.

In [ ]:
# Compute deviation = measured_zk - intrinsic(at WFS positions)
# Apply Z4 height correction to measured Zernikes first if available

n_wfs_pts = 4
wfs_deviation = np.full((n_visits, n_wfs_pts, n_j), np.nan)
wfs_measured = np.full((n_visits, n_wfs_pts, n_j), np.nan)
wfs_intrinsic = np.full((n_visits, n_wfs_pts, n_j), np.nan)

for v_idx in range(n_visits):
    r = wfs_results[v_idx]
    zk_meas = r['zk_median'].copy()  # (4, n_zk)

    # Apply Z4 height correction to measured Zernikes if available
    if Z4hgt is not None and 4 in iZidx:
        d, s = r['day_obs'], r['seq_num']
        visit_mask = (dobs_arr == d) & (snum_arr == s)

        # For Z4 correction of the WFS median: we need the median Z4hgt
        # in each wedge, matching how we compute the median Zernikes
        thx_donut = np.degrees(np.asarray(
            donut_df.loc[visit_mask, f'thx_{coord_sys}'], dtype=float))
        thy_donut = np.degrees(np.asarray(
            donut_df.loc[visit_mask, f'thy_{coord_sys}'], dtype=float))
        r_donut = np.hypot(thx_donut, thy_donut)
        az_donut = np.degrees(np.arctan2(thy_donut, thx_donut)) % 360.0
        z4hgt_visit = Z4hgt[visit_mask]
        z4hgt_intr_visit = Z4hgt_intrinsic[visit_mask]

        radial_mask = ((r_donut >= wfs_inner_radius_deg)
                       & (r_donut <= wfs_outer_radius_deg))
        half_az = wfs_azimuth_width_deg / 2.0
        wfs_centers_az = [(delta_deg + offset) % 360.0
                          for offset in [0, 90, 180, 270]]

        for i, az_center in enumerate(wfs_centers_az):
            az_lo = (az_center - half_az) % 360.0
            az_hi = (az_center + half_az) % 360.0
            if az_lo < az_hi:
                az_mask = (az_donut >= az_lo) & (az_donut <= az_hi)
            else:
                az_mask = (az_donut >= az_lo) | (az_donut <= az_hi)
            wedge = radial_mask & az_mask
            if wedge.sum() >= 3:
                # Correct measured Z4: subtract data Z4hgt
                z4_corr = np.nanmedian(z4hgt_visit[wedge])
                zk_meas[i, iZidx[4]] -= z4_corr

    # Intrinsic at WFS positions
    zk_intr = intrinsic_grid.evaluate(r['thx_deg'], r['thy_deg'])

    wfs_measured[v_idx] = zk_meas
    wfs_intrinsic[v_idx] = zk_intr
    wfs_deviation[v_idx] = zk_meas - zk_intr

print(f'WFS deviation array: {wfs_deviation.shape}  (n_visits, 4, n_zk)')
print(f'Deviation RMS per Zernike (across visits and WFS positions):')
for ji, j in enumerate(iZs):
    rms = np.sqrt(np.nanmean(wfs_deviation[:, :, ji]**2))
    print(f'  Z{j}: {rms:.4f} μm')

### WFS Deviation Example

Show measured, intrinsic, and deviation for one example visit at each
WFS position.

In [ ]:
# Bar chart: measured, intrinsic, deviation for example visit.
# Reuse ex_idx from the region plot (a visit with all 4 wedges populated),
# so every WFS panel — including WFS 0 — has measured/deviation bars and
# not just the intrinsic (which is always defined from the grid).
fig, axes = plt.subplots(2, 2, figsize=(14, 10), layout='constrained')
axes = axes.ravel()
j_labels = [f'Z{j}' for j in iZs]
x = np.arange(n_j)

for i in range(4):
    ax = axes[i]
    ax.bar(x - 0.25, wfs_measured[ex_idx, i], width=0.25,
           label='measured', color='tab:blue', alpha=0.8)
    ax.bar(x, wfs_intrinsic[ex_idx, i], width=0.25,
           label='intrinsic', color='tab:orange', alpha=0.8)
    ax.bar(x + 0.25, wfs_deviation[ex_idx, i], width=0.25,
           label='deviation', color='tab:green', alpha=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(j_labels, fontsize=7, rotation=45)
    ax.set_ylabel('μm')
    r = wfs_results[ex_idx]
    ax.set_title(f'WFS {i} — ({r["thx_deg"][i]:.2f}, {r["thy_deg"][i]:.2f})°, '
                 f'n={r["n_donuts"][i]}', fontsize=9)
    ax.axhline(0, color='k', lw=0.5)
    ax.grid(alpha=0.3)
    if i == 0:
        ax.legend(fontsize=8)

d, s = wfs_results[ex_idx]['day_obs'], wfs_results[ex_idx]['seq_num']
fig.suptitle(f'WFS Mimic — visit {d}/{s}, delta={delta_deg:.0f}°', fontsize=12)
plt.show()

<a id='svd'></a>
## 7. Sensitivity Matrix & SVD

We build two representations of the OFC sensitivity matrix:

1. **WFS-Zernike basis**: The sensitivity matrix evaluated directly at
   the 4 WFS field positions, giving a `(4 × n_zk, n_dof)` matrix.
   This is the natural basis for the WFS measurement vector.

2. **Double Zernike (DZ) basis**: The standard `(n_k × n_j, n_dof)`
   sensitivity matrix used by `build_measured_intrinsic`.

Both are right-multiplied by the OFC normalization matrix and SVD'd.
The key comparison is how the singular value spectrum and reachability
differ between the two representations — the WFS basis has fewer
measurements (84 vs 126) and loses information about the focal-plane
Zernike structure beyond what 4 points can resolve.

In [ ]:
# Auto-detect RSP location for OFC config
rsp_location = detect_rsp_location()
ofc_config_dir = get_packages_dir(rsp_location) + '/' + ofc_config_subpath
print(f'OFC config dir: {ofc_config_dir}')

ofc_data = OFCData('lsst', config_dir=ofc_config_dir)

# --- OFC Normalization weights ---
def _find_ofc_normalization_yaml(user_path=None,
                                 name='range0.5_fwhm-0.15.yaml'):
    if user_path:
        p = Path(user_path)
        if not p.exists():
            raise FileNotFoundError(f'{p} does not exist')
        return p
    root = os.environ.get('TS_CONFIG_MTTCS_DIR')
    if not root:
        raise RuntimeError(
            'TS_CONFIG_MTTCS_DIR is not set. Run setup ts_config_mttcs first.')
    target = Path(root) / 'MTAOS' / 'ofc' / 'normalization_weights' / name
    if not target.exists():
        raise FileNotFoundError(f'{target} not found')
    return target

norm_yaml_path = _find_ofc_normalization_yaml(ofc_normalization_yaml)
print(f'OFC normalization yaml: {norm_yaml_path}')
with open(norm_yaml_path) as _fp:
    normalization_weights = np.array(yaml.safe_load(_fp))
N_diag = normalization_weights
normalization_matrix = np.diag(normalization_weights)
print(f'Normalization: {len(normalization_weights)} weights '
      f'(min={normalization_weights.min():.3g}, max={normalization_weights.max():.3g})')

### WFS Sensitivity Matrix

Evaluate the OFC sensitivity matrix at the 4 WFS mimic field positions
using `SensitivityMatrix.evaluate()`.  This gives the wavefront
response at each WFS position for each DOF perturbation.

In [ ]:
# WFS field positions (thx, thy in degrees)
wfs_thx = wfs_results[0]['thx_deg']
wfs_thy = wfs_results[0]['thy_deg']
wfs_field_angles = [[float(wfs_thx[i]), float(wfs_thy[i])] for i in range(4)]
print('WFS field positions (deg):')
for i, (tx, ty) in enumerate(wfs_field_angles):
    r = np.hypot(tx, ty)
    az = np.degrees(np.arctan2(ty, tx))
    print(f'  WFS {i}: ({tx:+.4f}, {ty:+.4f}), r={r:.4f}°, az={az:.1f}°')

# Evaluate sensitivity matrix at WFS positions
dz_sens = SensitivityMatrix(ofc_data)
sens_3d = dz_sens.evaluate(wfs_field_angles, 0.0)
print(f'\nRaw sensitivity shape: {sens_3d.shape}  (n_wfs, n_zk_full, n_dof_full)')

# Slice to our Zernike selection
zn_idx = iZs_arr - 4
sens_wfs_3d = sens_3d[:, zn_idx, :]
n_dof = sens_wfs_3d.shape[2]
print(f'After Zernike selection: {sens_wfs_3d.shape}  (4, {n_j}, {n_dof})')

# Flatten to 2D: (4*n_j, n_dof) and apply normalization
S_wfs = sens_wfs_3d.reshape(-1, n_dof) @ normalization_matrix
print(f'S_wfs (normalized): {S_wfs.shape}')

# SVD of WFS sensitivity matrix
U_wfs, Sigma_wfs, Vh_wfs = np.linalg.svd(S_wfs, full_matrices=False)
V_wfs = Vh_wfs.T
n_keep_wfs = min(n_keep, U_wfs.shape[1])
U_wfs_eff = U_wfs[:, :n_keep_wfs]
print(f'SVD: U={U_wfs.shape}, Σ range=[{Sigma_wfs[-1]:.3g}, {Sigma_wfs[0]:.3g}], '
      f'V={V_wfs.shape}')
print(f'U_wfs_eff: {U_wfs_eff.shape}  (n_keep={n_keep_wfs})')

### DZ Sensitivity Matrix (reference)

Build the standard Double Zernike sensitivity matrix using the same
procedure as `build_measured_intrinsic`, for comparison.

In [ ]:
# DZ sensitivity matrix — from ofc_data.sensitivity_matrix
S_full = np.asarray(ofc_data.sensitivity_matrix)
print(f'S_full shape (field, pupil-j, DoF) = {S_full.shape}')

n_k = k_max - k_min + 1
S_slab = S_full[k_min:k_max + 1, iZs_arr, :]
S_dz = S_slab.reshape(-1, n_dof) @ normalization_matrix
print(f'S_dz (normalized): {S_dz.shape}  [k={k_min}..{k_max}, j={iZs_arr.min()}..{iZs_arr.max()}]')

kj_grid = [(int(k_min + ki), int(iZs_arr[ji]))
           for ki in range(n_k) for ji in range(n_j)]

U_dz, Sigma_dz, Vh_dz = np.linalg.svd(S_dz, full_matrices=False)
V_dz = Vh_dz.T
n_keep_dz = min(n_keep, U_dz.shape[1])
U_dz_eff = U_dz[:, :n_keep_dz]
print(f'SVD: U={U_dz.shape}, Σ range=[{Sigma_dz[-1]:.3g}, {Sigma_dz[0]:.3g}], '
      f'V={V_dz.shape}')

### SVD Diagnostic Plots

Compare the singular value spectra, V matrices, and sensitivity matrix
heatmaps for the WFS and DZ representations.

In [ ]:
# --- Singular value spectrum comparison ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5), layout='constrained')

ax = axes[0]
ax.semilogy(np.arange(1, len(Sigma_wfs) + 1), Sigma_wfs, 'o-', ms=4,
            label=f'WFS (4 pos, {S_wfs.shape[0]} rows)')
ax.semilogy(np.arange(1, len(Sigma_dz) + 1), Sigma_dz, 's-', ms=4,
            label=f'DZ (k={k_min}..{k_max}, {S_dz.shape[0]} rows)')
ax.axvline(n_keep_wfs, color='gray', ls='--', alpha=0.5,
           label=f'n_keep={n_keep_wfs}')
ax.set_xlabel('Mode index')
ax.set_ylabel('Singular value')
ax.set_title('Singular Value Spectrum')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

ax = axes[1]
ratio = Sigma_wfs[:min(len(Sigma_wfs), len(Sigma_dz))] / Sigma_dz[:min(len(Sigma_wfs), len(Sigma_dz))]
ax.plot(np.arange(1, len(ratio) + 1), ratio, 'o-', ms=4, color='tab:green')
ax.set_xlabel('Mode index')
ax.set_ylabel('σ_WFS / σ_DZ')
ax.set_title('Singular Value Ratio (WFS / DZ)')
ax.axhline(1, color='k', lw=0.5)
ax.grid(alpha=0.3)

fig.suptitle('Sensitivity Matrix SVD — WFS vs Double Zernike', fontsize=12)
plt.show()

In [ ]:
# --- Heatmap: S_wfs in WFS-Zernike basis (rows) vs v-modes (cols) ---
# Project onto v-modes: S_wfs @ V[:, :n_keep] / sigma gives U_eff
n_show = min(n_keep_wfs, 34)

fig, axes = plt.subplots(1, 2, figsize=(16, 8), layout='constrained')

# WFS Zernike vs v-mode
ax = axes[0]
wfs_row_labels = [f'WFS{s} Z{j}' for s in range(4) for j in iZs]
im = ax.imshow(U_wfs_eff[:, :n_show], aspect='auto', cmap='RdBu_r',
               vmin=-0.3, vmax=0.3)
ax.set_xlabel('v-mode index')
ax.set_ylabel('WFS Zernike row')
ax.set_title(f'U_wfs  ({S_wfs.shape[0]} rows × {n_show} modes)')
ax.set_yticks(np.arange(0, 4 * n_j, n_j))
ax.set_yticklabels([f'WFS{s}' for s in range(4)], fontsize=9)
for s in range(1, 4):
    ax.axhline(s * n_j - 0.5, color='k', lw=0.8)
plt.colorbar(im, ax=ax, shrink=0.8)

# DZ vs v-mode
ax = axes[1]
im = ax.imshow(U_dz_eff[:, :n_show], aspect='auto', cmap='RdBu_r',
               vmin=-0.3, vmax=0.3)
ax.set_xlabel('v-mode index')
ax.set_ylabel('DZ (k,j) row')
ax.set_title(f'U_dz  ({S_dz.shape[0]} rows × {n_show} modes)')
ax.set_yticks(np.arange(0, n_k * n_j, n_j))
ax.set_yticklabels([f'k={k_min+ki}' for ki in range(n_k)], fontsize=9)
for ki in range(1, n_k):
    ax.axhline(ki * n_j - 0.5, color='k', lw=0.8)
plt.colorbar(im, ax=ax, shrink=0.8)

fig.suptitle('Sensitivity Matrix U-modes — WFS vs DZ basis', fontsize=12)
plt.show()

In [ ]:
# --- V matrix comparison: WFS vs DZ ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6), layout='constrained')

for ax_idx, (V_mat, label) in enumerate([
        (V_wfs, 'V_wfs'), (V_dz, 'V_dz')]):
    ax = axes[ax_idx]
    n_show_v = min(n_keep, V_mat.shape[1])
    im = ax.imshow(V_mat[:, :n_show_v], aspect='auto', cmap='RdBu_r',
                   vmin=-0.3, vmax=0.3)
    ax.set_xlabel('v-mode index')
    ax.set_ylabel('DOF index')
    ax.set_title(f'{label}  ({V_mat.shape[0]} DOFs × {n_show_v} modes)')
    ax.set_yticks([0, 5, 10, 30])
    ax.set_yticklabels(['M2_dz', 'Cam_dz', 'B1_1', 'B2_1'], fontsize=8)
    ax.axhline(9.5, color='k', lw=0.5, alpha=0.5)
    ax.axhline(29.5, color='k', lw=0.5, alpha=0.5)
    plt.colorbar(im, ax=ax, shrink=0.8)

fig.suptitle('V matrices (DOF space) — WFS vs DZ', fontsize=12)
plt.show()

<a id='dof-recovery'></a>
## 8. DOF Recovery from WFS Mimic

For each visit, project the WFS deviation vector onto the WFS U-modes
and recover the physical DOFs.  We also recover DOFs using the DZ
representation for the same WFS measurements, to compare the two
projection paths.

In [ ]:
# Flatten WFS deviation to (n_visits, 4*n_j)
w_wfs_flat = wfs_deviation.reshape(n_visits, -1)

# A visit only yields a meaningful WFS solution if its full deviation
# vector is finite.  Any NaN (a wedge with <3 donuts, or a WFS position
# outside the intrinsic grid) leaves A_modes NaN, which recover_dof_per_visit
# zeros out -> the visit comes back as an all-zero DOF vector.  Flag those
# visits explicitly and exclude them rather than letting the zeros pollute
# the WFS-vs-FAM comparison.
wfs_finite = np.all(np.isfinite(w_wfs_flat), axis=1)

# Project onto WFS U-modes (only for finite visits)
A_modes_wfs = np.full((n_visits, n_keep_wfs), np.nan)
for v in range(n_visits):
    if wfs_finite[v]:
        A_modes_wfs[v] = U_wfs_eff.T @ w_wfs_flat[v]

# Recover DOFs
dof_wfs = recover_dof_per_visit(
    A_modes_wfs, V_wfs, Sigma_wfs, N_diag, n_keep_wfs)

# Identify and filter out invalid / all-zero visits
all_zero = np.all(dof_wfs == 0, axis=1)
wfs_invalid = (~wfs_finite) | all_zero
dof_wfs[wfs_invalid] = np.nan

print(f'WFS DOF array: {dof_wfs.shape}  (n_visits, n_dof)')

# Count valid
n_valid = np.sum(np.all(np.isfinite(dof_wfs), axis=1))
print(f'Valid visits: {n_valid}/{n_visits}')

# Report flagged visits and the reason each one collapsed to zero
n_flagged = int(wfs_invalid.sum())
print(f'\nFlagged (all-zero / invalid) visits: {n_flagged}/{n_visits}')
if n_flagged:
    print('  day_obs/seq_num : reason')
    for v in np.where(wfs_invalid)[0]:
        d = int(dz_df.iloc[v]['day_obs'])
        s = int(dz_df.iloc[v]['seq_num'])
        cnts = all_counts[v]
        bad_wedges = np.where(cnts < 3)[0]
        # WFS positions where the deviation is non-finite (measured or intrinsic NaN)
        bad_dev = np.where(~np.all(np.isfinite(wfs_deviation[v]), axis=1))[0]
        intr_nan = np.where(~np.all(np.isfinite(wfs_intrinsic[v]), axis=1))[0]
        reasons = []
        if bad_wedges.size:
            reasons.append(f'WFS{bad_wedges.tolist()} <3 donuts (counts={cnts.tolist()})')
        if intr_nan.size:
            reasons.append(f'intrinsic NaN at WFS{intr_nan.tolist()}')
        if not reasons and bad_dev.size:
            reasons.append(f'non-finite deviation at WFS{bad_dev.tolist()}')
        if not reasons:
            reasons.append('all-zero solution (degenerate)')
        print(f'  {d}/{s} : ' + '; '.join(reasons))

### FAM DOFs (from build_measured_intrinsic)

Extract the FAM-derived DOFs from the dz_fits parquet for comparison.

In [ ]:
# Extract FAM DOFs from dz_fits
dof_fam = np.full((n_visits, 50), np.nan)
for di, name in enumerate(LABELS_50DOF):
    col = f'dof_{name}'
    if col in dz_df.columns:
        dof_fam[:, di] = dz_df[col].values

print(f'FAM DOF array: {dof_fam.shape}')
n_valid_fam = np.sum(np.all(np.isfinite(dof_fam), axis=1))
print(f'Valid FAM visits: {n_valid_fam}/{n_visits}')

<a id='comparison'></a>
## 9. Comparison: WFS vs FAM DOFs

The key diagnostic: how well do the WFS-derived DOFs track the
FAM-derived DOFs?  We show:
1. DOF time series — WFS vs FAM overlaid
2. Scatter plots — WFS DOF vs FAM DOF for each component
3. Correlation coefficients across the 50 DOFs

In [ ]:
# --- DOF vs visit: WFS and FAM overlaid ---
# 4-panel layout: Hex Translations, Hex Rotations, M1M3, M2

hex_trans_idx = [0, 1, 2, 5, 6, 7]
hex_rot_idx = [3, 4, 8, 9]
m1m3_idx = list(range(10, 30))
m2_idx = list(range(30, 50))

groups = [
    ('Hexapod Translations (μm)', hex_trans_idx),
    ('Hexapod Rotations (arcsec)', hex_rot_idx),
    ('M1M3 Bending Modes (μm)', m1m3_idx),
    ('M2 Bending Modes (μm)', m2_idx),
]

ordinal = np.arange(n_visits)

for group_title, dof_indices in groups:
    n_panels = len(dof_indices)
    ncols = min(5, n_panels)
    nrows = (n_panels + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols,
                             figsize=(3.5 * ncols, 2.5 * nrows),
                             layout='constrained', sharex=True)
    axes = np.atleast_1d(axes).ravel()

    for pidx, di in enumerate(dof_indices):
        ax = axes[pidx]
        y_fam = dof_fam[:, di]
        y_wfs = dof_wfs[:, di]
        ax.plot(ordinal, y_fam, '.', ms=3, alpha=0.7, color='tab:blue',
                label='FAM')
        ax.plot(ordinal, y_wfs, '.', ms=3, alpha=0.7, color='tab:red',
                label='WFS')
        ax.set_title(f'{LABELS_50DOF[di]}', fontsize=8)
        ax.axhline(0, color='k', lw=0.4, alpha=0.5)
        ax.grid(alpha=0.3)
        ax.tick_params(labelsize=7)
        if pidx == 0:
            ax.legend(fontsize=7)

    for k in range(n_panels, len(axes)):
        axes[k].set_visible(False)

    fig.suptitle(f'{group_title} — WFS (red) vs FAM (blue)', fontsize=12)
    plt.show()

In [ ]:
# --- Scatter plots: WFS vs FAM DOF ---
fig, axes = plt.subplots(5, 10, figsize=(22, 12), layout='constrained')
axes = axes.ravel()

correlations = np.full(50, np.nan)
for di in range(50):
    ax = axes[di]
    x = dof_fam[:, di]
    y = dof_wfs[:, di]
    good = np.isfinite(x) & np.isfinite(y)
    if good.sum() >= 5:
        ax.plot(x[good], y[good], '.', ms=2, alpha=0.5)
        corr = np.corrcoef(x[good], y[good])[0, 1]
        correlations[di] = corr
        # 1:1 line
        lims = [min(x[good].min(), y[good].min()),
                max(x[good].max(), y[good].max())]
        ax.plot(lims, lims, 'k-', lw=0.5, alpha=0.5)
        ax.set_title(f'{LABELS_50DOF[di]}\nr={corr:.2f}', fontsize=6)
    else:
        ax.set_title(f'{LABELS_50DOF[di]}\n(no data)', fontsize=6)
    ax.tick_params(labelsize=5)
    ax.set_aspect('equal', adjustable='datalim')

fig.suptitle('WFS DOF vs FAM DOF — per component scatter + correlation',
             fontsize=12)
fig.supxlabel('FAM DOF', fontsize=10)
fig.supylabel('WFS DOF', fontsize=10)
plt.show()

In [ ]:
# --- Summary: correlation bar chart ---
fig, ax = plt.subplots(figsize=(16, 5), layout='constrained')
x_pos = np.arange(50)
colors_bar = np.where(correlations > 0.7, 'tab:green',
              np.where(correlations > 0.3, 'tab:orange', 'tab:red'))
ax.bar(x_pos, correlations, color=colors_bar, alpha=0.8)
ax.set_xticks(x_pos)
ax.set_xticklabels(LABELS_50DOF, rotation=90, fontsize=7)
ax.set_ylabel('Pearson r (WFS vs FAM)')
ax.set_title('DOF Recovery Correlation: WFS Mimic vs FAM')
ax.axhline(0.7, color='green', ls='--', lw=0.8, alpha=0.5, label='r=0.7')
ax.axhline(0.3, color='orange', ls='--', lw=0.8, alpha=0.5, label='r=0.3')
ax.axhline(0, color='k', lw=0.5)
ax.axvline(9.5, color='k', lw=0.5, alpha=0.3)
ax.axvline(29.5, color='k', lw=0.5, alpha=0.3)
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)

n_good = np.sum(correlations > 0.7)
n_fair = np.sum((correlations > 0.3) & (correlations <= 0.7))
n_poor = np.sum(correlations <= 0.3)
ax.text(0.98, 0.95,
        f'r>0.7: {n_good}  |  0.3<r≤0.7: {n_fair}  |  r≤0.3: {n_poor}',
        transform=ax.transAxes, ha='right', va='top', fontsize=10,
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
plt.show()

print(f'\nSummary: {n_good}/50 DOFs well-recovered (r>0.7), '
      f'{n_fair} fair, {n_poor} poor')

### DOF Residual Analysis

The residual (WFS - FAM) DOF shows the systematic bias and scatter
introduced by using only 4 field positions.  Large residuals indicate
DOFs that are poorly constrained by the WFS geometry.

In [ ]:
# --- Residual: WFS - FAM ---
dof_residual = dof_wfs - dof_fam

fig, axes = plt.subplots(4, 1, figsize=(15, 14), layout='constrained',
                         gridspec_kw=dict(height_ratios=[1, 1, 1.5, 1.5]))

for ax_idx, (title, idx_list, unit) in enumerate([
        ('Hexapod Translations', hex_trans_idx, 'μm'),
        ('Hexapod Rotations', hex_rot_idx, 'arcsec'),
        ('M1M3 Bending Modes', m1m3_idx, 'μm'),
        ('M2 Bending Modes', m2_idx, 'μm')]):
    ax = axes[ax_idx]
    n_d = len(idx_list)
    x = np.arange(n_d)

    means = np.array([np.nanmean(dof_residual[:, di]) for di in idx_list])
    stds = np.array([np.nanstd(dof_residual[:, di]) for di in idx_list])

    ax.bar(x - 0.2, means, width=0.4, color='tab:blue',
           alpha=0.8, label='mean')
    ax.bar(x + 0.2, stds, width=0.4, color='tab:orange',
           alpha=0.8, label='std')
    ax.set_xticks(x)
    ax.set_xticklabels([LABELS_50DOF[di] for di in idx_list],
                       rotation=45, ha='right', fontsize=8)
    ax.set_ylabel(f'Residual ({unit})')
    ax.set_title(f'{title} — WFS − FAM residual')
    ax.axhline(0, color='k', lw=0.5)
    ax.grid(axis='y', alpha=0.3)
    ax.legend(fontsize=8)

fig.suptitle('DOF Residual (WFS Mimic − FAM): Mean ± Std', fontsize=12)
plt.show()